### Our goal is to determine whether structure-based chunking (EXP_002_v2) retrieves more relevant information than recursive character chunking (EXP_001).

Experiment design
We'll keep the following settings identical for both experiments:
Parameter	Setting
Embedding model	Same model
Retrieval method	Dense retrieval
Similarity metric	Same
Top-K	Same
Test questions	Same golden test set
Evaluation metrics	Recall@K, MRR, Precision@K, NDCG@K

We'll test dense retrieval first. Later, we can compare BM25 and hybrid retrieval separately.

In [3]:
# CELL 1: Locate chunking experiment files and golden dataset

from pathlib import Path

for folder in [
    Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\processed"),
    Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits")
]:
    print(f"\n--- {folder} ---")

    if folder.exists():
        for file in folder.rglob("*"):
            if file.is_file() and (
                file.name.endswith(".jsonl")
                or file.name.endswith(".json")
                or file.name.endswith(".csv")
            ):
                print(file)
    else:
        print("Folder not found")


--- G:\SEVA_Ai\SEVA_Ai\notebooks\data\processed ---
G:\SEVA_Ai\SEVA_Ai\notebooks\data\processed\child_documents.jsonl
G:\SEVA_Ai\SEVA_Ai\notebooks\data\processed\parent_documents.jsonl
G:\SEVA_Ai\SEVA_Ai\notebooks\data\processed\EXP_002_structure_based\children.jsonl
G:\SEVA_Ai\SEVA_Ai\notebooks\data\processed\EXP_002_structure_based\parents.jsonl
G:\SEVA_Ai\SEVA_Ai\notebooks\data\processed\EXP_002_v2\children.jsonl

--- G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\final\Final_dataset_splits ---
G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\final\Final_dataset_splits\test.json
G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\final\Final_dataset_splits\train.json
G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\final\Final_dataset_splits\validation.json


In [4]:
# CELL 3: Load and verify chunking datasets

import json
import pandas as pd
from pathlib import Path

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Load experiment datasets
exp_001 = load_jsonl("data/processed/child_documents.jsonl")

exp_002_v2 = load_jsonl(
    "data/processed/EXP_002_v2/children.jsonl"
)

# Load golden test set
with open(
    "golden_dataset/final/Final_dataset_splits/test.json",
    "r",
    encoding="utf-8"
) as f:
    test_data = json.load(f)

# Summary
print("EXP_001 chunks:", len(exp_001))
print("EXP_002_v2 chunks:", len(exp_002_v2))
print("Golden test questions:", len(test_data))

print("\nEXP_001 fields:", list(exp_001[0].keys()))
print("EXP_002_v2 fields:", list(exp_002_v2[0].keys()))
print("Golden dataset fields:", list(test_data[0].keys()))

# Inspect sample records
print("\nEXP_001 sample:")
print(json.dumps(exp_001[0], indent=2, ensure_ascii=False)[:1500])

print("\nEXP_002_v2 sample:")
print(json.dumps(exp_002_v2[0], indent=2, ensure_ascii=False)[:1500])

EXP_001 chunks: 124029
EXP_002_v2 chunks: 91858
Golden test questions: 973

EXP_001 fields: ['text', 'metadata']
EXP_002_v2 fields: ['child_id', 'parent_id', 'scheme_id', 'section', 'text', 'categories', 'document_type']
Golden dataset fields: ['test_id', 'faq_id', 'scheme_id', 'scheme_name', 'question', 'ground_truth_answer', 'categories', 'level', 'source_url', 'relevant_scheme_ids', 'relevant_doc_ids', 'answerable']

EXP_001 sample:
{
  "text": "Scheme: Stand-Up India\nSection: brief_description\nA scheme by Ministry of Finance for financing SC/ST and Women Entrepreneurs by facilitating bank loans for setting up a greenfield project Enterprise in manufacturing, services, trading sector and activities allied to agriculture.",
  "metadata": {
    "parent_id": "LBezF5gB5uF80_MGGY81_brief_description",
    "scheme_id": "LBezF5gB5uF80_MGGY81",
    "section": "brief_description",
    "document_type": "scheme_section",
    "categories": [
      "Business & Entrepreneurship",
      "Banking

In [5]:
# CELL 4: Normalize both chunking datasets

def normalize_chunks(records, experiment):
    normalized = []

    for record in records:
        metadata = record.get("metadata", {})

        # EXP_001 stores metadata inside "metadata".
        # EXP_002_v2 stores metadata at the top level.
        if not metadata:
            metadata = record

        normalized.append({
            "experiment": experiment,
            "child_id": metadata.get("child_id"),
            "parent_id": metadata.get("parent_id"),
            "scheme_id": metadata.get("scheme_id"),
            "section": metadata.get("section"),
            "document_type": metadata.get("document_type"),
            "categories": metadata.get("categories", []),
            "text": record.get("text", "")
        })

    return normalized


# Normalize
chunks_001 = normalize_chunks(exp_001, "EXP_001")
chunks_002 = normalize_chunks(exp_002_v2, "EXP_002_v2")

# Convert to DataFrames
df_001 = pd.DataFrame(chunks_001)
df_002 = pd.DataFrame(chunks_002)

# Validate
for name, df in [
    ("EXP_001", df_001),
    ("EXP_002_v2", df_002)
]:
    print(f"\n--- {name} ---")
    print("Total chunks:", len(df))
    print("Unique schemes:", df["scheme_id"].nunique())
    print("Missing scheme IDs:", df["scheme_id"].isna().sum())
    print("Missing text:", df["text"].isna().sum())
    print("Empty text:", (df["text"].str.strip() == "").sum())
    print("Duplicate child IDs:", df["child_id"].duplicated().sum())
    print("\nChunks by section:")
    print(df["section"].value_counts())


# Check golden test scheme coverage
test_scheme_ids = {
    sid
    for item in test_data
    for sid in item["relevant_scheme_ids"]
}

print("\n--- Golden test coverage ---")
print("Unique relevant schemes:", len(test_scheme_ids))
print("EXP_001 missing schemes:",
      len(test_scheme_ids - set(df_001["scheme_id"].dropna())))
print("EXP_002_v2 missing schemes:",
      len(test_scheme_ids - set(df_002["scheme_id"].dropna())))


--- EXP_001 ---
Total chunks: 124029
Unique schemes: 4428
Missing scheme IDs: 0
Missing text: 0
Empty text: 0
Duplicate child IDs: 0

Chunks by section:
section
faq                    48695
application_process    26139
details                12354
eligibility            11168
benefits               11116
documents_required      8887
brief_description       4428
exclusions              1242
Name: count, dtype: int64

--- EXP_002_v2 ---
Total chunks: 91858
Unique schemes: 4428
Missing scheme IDs: 0
Missing text: 0
Empty text: 0
Duplicate child IDs: 0

Chunks by section:
section
faq                    48935
application_process    12532
benefits                6590
details                 6507
eligibility             6481
documents_required      5481
brief_description       4428
exclusions               904
Name: count, dtype: int64

--- Golden test coverage ---
Unique relevant schemes: 473
EXP_001 missing schemes: 0
EXP_002_v2 missing schemes: 0


In [6]:
# CELL 5: Prepare documents for embedding

from langchain_core.documents import Document

def create_documents(df):
    documents = []

    for row in df.to_dict("records"):
        documents.append(
            Document(
                page_content=row["text"],
                metadata={
                    "child_id": row["child_id"],
                    "parent_id": row["parent_id"],
                    "scheme_id": row["scheme_id"],
                    "section": row["section"],
                    "document_type": row["document_type"],
                    "categories": row["categories"],
                    "experiment": row["experiment"]
                }
            )
        )

    return documents


docs_001 = create_documents(df_001)
docs_002 = create_documents(df_002)

print("EXP_001 documents:", len(docs_001))
print("EXP_002_v2 documents:", len(docs_002))

print("\nSample metadata:")
print(docs_001[0].metadata)

print("\nSample text:")
print(docs_001[0].page_content[:300])

EXP_001 documents: 124029
EXP_002_v2 documents: 91858

Sample metadata:
{'child_id': 'LBezF5gB5uF80_MGGY81_brief_description_CHILD_001', 'parent_id': 'LBezF5gB5uF80_MGGY81_brief_description', 'scheme_id': 'LBezF5gB5uF80_MGGY81', 'section': 'brief_description', 'document_type': 'scheme_section', 'categories': ['Business & Entrepreneurship', 'Banking,Financial Services and Insurance', 'Social welfare & Empowerment'], 'experiment': 'EXP_001'}

Sample text:
Scheme: Stand-Up India
Section: brief_description
A scheme by Ministry of Finance for financing SC/ST and Women Entrepreneurs by facilitating bank loans for setting up a greenfield project Enterprise in manufacturing, services, trading sector and activities allied to agriculture.


In [13]:
# # We'll use two independent collections to avoid mixing embeddings or documents between experiments.

# # Important: We will use the same embedding model instance for both collections and persist each collection separately.

# # CELL 6: Create separate Chroma vector indexes

# from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_chroma import Chroma
# from pathlib import Path

# # Same embedding model for both experiments
# embedding_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2",
#     model_kwargs={"device": "cpu"},
#     encode_kwargs={"normalize_embeddings": True}
# )

# # Separate persistent directories
# db_dir = Path("data/vectorstores/chunking_comparison")
# db_dir.mkdir(parents=True, exist_ok=True)

# # EXP_001 index
# vectorstore_001 = Chroma.from_documents(
#     documents=docs_001,
#     embedding=embedding_model,
#     collection_name="exp_001_recursive",
#     persist_directory=str(db_dir / "exp_001")
# )

# # EXP_002_v2 index
# vectorstore_002 = Chroma.from_documents(
#     documents=docs_002,
#     embedding=embedding_model,
#     collection_name="exp_002_structure_v2",
#     persist_directory=str(db_dir / "exp_002_v2")
# )

# print("Both Chroma indexes created successfully.")
# print("EXP_001:", vectorstore_001._collection.count())
# print("EXP_002_v2:", vectorstore_002._collection.count())

In [10]:
import platform
print(platform.processor())

Intel64 Family 6 Model 58 Stepping 9, GenuineIntel


In [11]:
# Benchmark embedding speed

import time

sample_docs = docs_001[:1000]
sample_texts = [doc.page_content for doc in sample_docs]

start = time.time()

vectors = embedding_model.embed_documents(sample_texts)

elapsed = time.time() - start

print("Documents:", len(sample_texts))
print("Time:", round(elapsed, 2), "seconds")
print("Documents per second:", round(len(sample_texts) / elapsed, 2))
print("Vector dimensions:", len(vectors[0]))

Documents: 1000
Time: 185.06 seconds
Documents per second: 5.4
Vector dimensions: 384


In [12]:
import time

sample_texts = [doc.page_content for doc in docs_001[:500]]

for batch_size in [16, 32, 64, 128]:

    embedding_model.encode_kwargs["batch_size"] = batch_size

    start = time.time()

    vectors = embedding_model.embed_documents(sample_texts)

    elapsed = time.time() - start

    print(
        f"Batch: {batch_size} | "
        f"Time: {elapsed:.2f}s | "
        f"Speed: {500 / elapsed:.2f} chunks/sec"
    )

Batch: 16 | Time: 79.86s | Speed: 6.26 chunks/sec
Batch: 32 | Time: 100.45s | Speed: 4.98 chunks/sec
Batch: 64 | Time: 111.16s | Speed: 4.50 chunks/sec
Batch: 128 | Time: 123.60s | Speed: 4.05 chunks/sec


In [ ]:
# ============================================================
# SEVA_AI: RESUMABLE BATCH EMBEDDING + CHROMA INDEXING
# EXP_001 vs EXP_002_v2
# ============================================================

import os
import json
import time
import torch
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

torch.set_num_threads(4)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

EMBEDDING_BATCH_SIZE = 16
INDEX_BATCH_SIZE = 1000

BASE_DIR = Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\vectorstores\\chunking_comparison")
BASE_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = {
    "EXP_001": {
        "documents": docs_001,
        "collection": "exp_001_recursive",
        "directory": BASE_DIR / "exp_001"
    },
    "EXP_002_v2": {
        "documents": docs_002,
        "collection": "exp_002_structure_v2",
        "directory": BASE_DIR / "exp_002_v2"
    }
}


# ------------------------------------------------------------
# 2. LOAD EMBEDDING MODEL
# ------------------------------------------------------------

embedding_model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

embedding_model.max_seq_length = 256

embedding_model.eval()

print("Embedding model loaded.")
print("Embedding dimensions:", embedding_model.get_sentence_embedding_dimension())


# ------------------------------------------------------------
# 3. RESUMABLE INDEXING FUNCTION
# ------------------------------------------------------------

def build_resumable_index(experiment_name, config):

    documents = config["documents"]
    collection_name = config["collection"]
    persist_dir = config["directory"]

    persist_dir.mkdir(parents=True, exist_ok=True)

    progress_file = persist_dir / "progress.json"

    print(f"\n{'='*55}")
    print(f"Starting {experiment_name}")
    print(f"Total documents: {len(documents)}")
    print(f"{'='*55}")

    # Create Chroma collection
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=None,
        persist_directory=str(persist_dir)
    )

    collection = vectorstore._collection

    # Verify collection belongs to this dataset and model.
    dataset_ids = [
        doc.metadata["child_id"]
        for doc in documents
    ]

    if len(dataset_ids) != len(set(dataset_ids)):
        raise ValueError(f"{experiment_name}: Duplicate child IDs found.")

    dataset_signature = {
        "experiment": experiment_name,
        "model": MODEL_NAME,
        "max_seq_length": embedding_model.max_seq_length,
        "total_documents": len(documents),
        "first_id": dataset_ids[0],
        "last_id": dataset_ids[-1]
    }

    if progress_file.exists():

        with open(progress_file, "r", encoding="utf-8") as f:
            progress = json.load(f)

        if progress["signature"] != dataset_signature:
            raise ValueError(
                f"{experiment_name}: Existing progress belongs to "
                "a different dataset/configuration. Use a new directory."
            )

        print("Existing progress found. Resuming...")

    else:

        progress = {
            "signature": dataset_signature,
            "completed_ids": []
        }

        with open(progress_file, "w", encoding="utf-8") as f:
            json.dump(progress, f, indent=2)

    completed_ids = set(progress["completed_ids"])

    # Also check Chroma for records already inserted.
    chroma_ids = set()

    total_in_collection = collection.count()

    for start in range(0, total_in_collection, 10000):

        result = collection.get(
            limit=10000,
            offset=start,
            include=[]
        )

        chroma_ids.update(result["ids"])

    completed_ids.update(chroma_ids)

    print("Already completed:", len(completed_ids))

    # Prepare remaining documents
    remaining_docs = [
        doc for doc in documents
        if doc.metadata["child_id"] not in completed_ids
    ]

    print("Remaining documents:", len(remaining_docs))

    # Process in batches
    start_time = time.time()

    for start in tqdm(
        range(0, len(remaining_docs), INDEX_BATCH_SIZE),
        desc=experiment_name
    ):

        batch_docs = remaining_docs[
            start:start + INDEX_BATCH_SIZE
        ]

        texts = [
            doc.page_content
            for doc in batch_docs
        ]

        ids = [
            doc.metadata["child_id"]
            for doc in batch_docs
        ]

        metadatas = [
            {
                "parent_id": doc.metadata.get("parent_id") or "",
                "scheme_id": doc.metadata.get("scheme_id") or "",
                "section": doc.metadata.get("section") or "",
                "document_type": doc.metadata.get("document_type") or "",
                "categories": json.dumps(
                    doc.metadata.get("categories", []),
                    ensure_ascii=False
                ),
                "experiment": experiment_name
            }
            for doc in batch_docs
        ]

        # Generate embeddings in smaller CPU batches
        embeddings = embedding_model.encode(
            texts,
            batch_size=EMBEDDING_BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        # Insert vectors into Chroma
        collection.upsert(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadatas
        )

        # Save progress only after successful insertion
        completed_ids.update(ids)

        progress["completed_ids"] = list(completed_ids)

        with open(progress_file, "w", encoding="utf-8") as f:
            json.dump(progress, f)

    elapsed = time.time() - start_time

    # Final validation
    final_count = collection.count()

    print(f"\n{experiment_name} completed.")
    print("Expected documents:", len(documents))
    print("Indexed documents:", final_count)
    print("Time taken:", round(elapsed / 60, 2), "minutes")

    if final_count != len(documents):
        raise ValueError(
            f"{experiment_name}: Index count mismatch!"
        )

    return vectorstore


# ------------------------------------------------------------
# 4. BUILD BOTH EXPERIMENT INDEXES
# ------------------------------------------------------------

vectorstore_001 = build_resumable_index(
    "EXP_001",
    EXPERIMENTS["EXP_001"]
)

vectorstore_002 = build_resumable_index(
    "EXP_002_v2",
    EXPERIMENTS["EXP_002_v2"]
)


# ------------------------------------------------------------
# 5. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "="*55)
print("FINAL INDEXING SUMMARY")
print("="*55)

print(
    "EXP_001:",
    vectorstore_001._collection.count()
)

print(
    "EXP_002_v2:",
    vectorstore_002._collection.count()
)

print("\nBoth vector indexes are ready for retrieval evaluation.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 885.78it/s]


Embedding model loaded.
Embedding dimensions: 384

Starting EXP_001
Total documents: 124029


C:\Users\devendra\AppData\Local\Temp\ipykernel_1572\3855422282.py:61: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", embedding_model.get_sentence_embedding_dimension())


Already completed: 0
Remaining documents: 124029


EXP_001:  12%|█▏        | 15/125 [42:29<5:14:26, 171.51s/it]